# 🎨 Brain Flower Recoloring with SVG Palette
Ce notebook montre comment extraire une palette de couleurs à partir d'un fichier SVG et l'utiliser pour recolorer une illustration artistique (cerveau fleuri).

In [ ]:
# 📦 Imports
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cairosvg
from io import BytesIO
from sklearn.cluster import KMeans

## 1. Chargement et conversion du fichier SVG

In [ ]:
svg_path = 'blob-scene-haikei.svg'
png_output = BytesIO()
cairosvg.svg2png(url=svg_path, write_to=png_output)
png_output.seek(0)
svg_image = Image.open(png_output).convert('RGB')

## 2. Extraction des couleurs dominantes du SVG

In [ ]:
small_svg = svg_image.resize((100, 100))
svg_pixels = np.array(small_svg).reshape(-1, 3)
kmeans_svg = KMeans(n_clusters=5, random_state=0).fit(svg_pixels)
svg_colors = kmeans_svg.cluster_centers_.astype(int)

# Affichage des couleurs
fig, ax = plt.subplots(1, 6, figsize=(12, 2))
ax[0].imshow(small_svg)
ax[0].axis('off')
ax[0].set_title('SVG')
for i, color in enumerate(svg_colors):
    patch = np.zeros((100, 100, 3), dtype=int)
    patch[:, :] = color
    ax[i+1].imshow(patch.astype(np.uint8))
    ax[i+1].axis('off')
    ax[i+1].set_title(f'Color {i+1}')
plt.tight_layout()
plt.show()

## 3. Chargement de l’image du cerveau fleuri

In [ ]:
brain_path = 'ChatGPT Image 10 avr. 2025, 22_08_01.png'
brain_img = Image.open(brain_path).convert('RGB')
brain_array = np.array(brain_img)
reshaped_brain = brain_array.reshape(-1, 3)

## 4. Analyse des couleurs de l’image du cerveau

In [ ]:
kmeans_brain = KMeans(n_clusters=5, random_state=1).fit(reshaped_brain)
brain_colors = kmeans_brain.cluster_centers_.astype(int)

## 5. Mapping des couleurs entre le cerveau et le SVG

In [ ]:
color_mapping = {tuple(brain_colors[i]): tuple(svg_colors[i]) for i in range(5)}

def map_color(pixel):
    for original, new_color in color_mapping.items():
        if np.allclose(pixel, original, atol=30):
            return new_color
    return pixel

new_pixels = np.array([map_color(pixel) for pixel in reshaped_brain], dtype=np.uint8)
new_image_array = new_pixels.reshape(brain_array.shape)
new_image = Image.fromarray(new_image_array)

## 6. Sauvegarde et affichage du résultat final

In [ ]:
output_path = 'brain_flower_recolored.png'
new_image.save(output_path)
new_image.show()